In [2]:
import tiktoken
from parsivar import Normalizer
import pandas as pd
import json
import shortuuid
import regex as re
import openai
import time
# from openai.embeddings_utils import get_embedding, cosine_similarity;
# import matplotlib


In [3]:
uuid = shortuuid.ShortUUID(alphabet="0123456789abcdefghijklmnopqrstuvwxyz")
enc = tiktoken.get_encoding("cl100k_base")
pd.set_option('display.max_rows', None)


In [4]:
with open('./majid/result.json', encoding='utf-8') as file:
    maj = json.loads(file.read())
len(maj)

4

In [5]:
with open('result.json', encoding='utf-8') as file:
    result = json.loads(file.read())
len(result)

4

In [6]:
stocks=pd.read_csv('stocks/stocks_tsetmcs.csv')
stocks=stocks[["sym_name","name","sym","gp_name"]]
stocks.head()

,sym_name,name,sym,gp_name
0,18719101,اوراق مشاركت ايران‌ خودرو,IKCQ1,خودرو و ساخت قطعات
1,آباد,توريستي ورفاهي آبادگران ايران,ABAD1,هتل و رستوران
2,آبادا,توليد نيروي برق آبادان,NBAB1,عرضه برق، گاز، بخاروآب گرم
3,آبادح,ح . ‌توريستي‌ورفاهي‌آبادگران‌,ABAX1,هتل و رستوران
4,آپ,آسان پرداخت پرشين,APPE1,رايانه و فعاليت‌هاي وابسته به آن


In [7]:
# txt="""# فولاژ
# در آستانه صف خرید به دلیل رشد 50 درصدی متوقف شد
#     """

# with open('./stocks/stk_id.json', encoding='utf-8') as file:
#     stocks = json.loads(file.read())
# len(stocks)
# stocks

def norm(txt):
    return Normalizer().normalize(txt)

def token_count(txt):
    return len(enc.encode(txt))

def if_chatlist_id_included(chatlist,ID):
#     print("CHATLIS__",chat_list)
    for chat in chatlist:
        if ID in chat['id_list']:
            return True
    return False

def if_chatlist_id_included2(chatlist,ID):
#     print("CHATLIS__",chat_list)
    for chat in chatlist:
        if ID == chat['chat_id']:
            return True
    return False

def if_stock_included(txt):
    for index, row in stocks.iterrows():
        if norm(row["sym_name"]) in norm(txt):
#             print("if_stock_included__")
#             print(f"_*_{norm(txt)}_*_",row["sym_name"])
            return True
    return  False

#if stock name is included in the text it will be replaced by stocks ID RETURNS: (if_is_included,text)
def if_stock_included_replace(txt): 
    txt=norm(txt)
#     txt_list=txt.split()

    for index, row in stocks.iterrows():
        norm_sym=norm(row['sym_name']).replace('\u200c', ' ')
        txt=re.sub(fr'\b{norm_sym}\b', f"<{row['sym']}>", txt)
#     print("replace_stock_with_idـ",norm_sym)
    if re.findall(r'<[0-9A-Za-z]>*', txt):
        return True,txt
    return False,txt
    

def clean_text(txt):
    replacement_patterns = {
            r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+': '',# http website
            r'[\w\.-]+@[\w\.-]+': '',# email
            r'@[A-Za-z0-9]*':'',# @ID
            r'#':'',
            r'\n{2,}':'\n'
            }
    for pattern, replacement in replacement_patterns.items():
            txt = re.sub(pattern, replacement, txt)
    return txt



In [8]:
with open('./majid/result.json', encoding='utf-8') as file:
    maj = json.loads(file.read())
# maj

### Preprocessing channel

In [9]:
%%time

messages=maj['messages']
chat_list_2 = []
for mess in messages:
    txt = ''
    chat_info={"person_id":mess['from_id'],"chat_id":mess['id']}

    for tx in mess['text_entities']:
        txt+=clean_text(tx['text']).strip()+'\n'
#     chat_info["text"]= txt
    is_stock_included,txt = if_stock_included_replace(txt.strip())
    
    if ("reply_to_message_id" in mess) and ( if_chatlist_id_included2(chat_list_2 , mess["reply_to_message_id"])):
        chat_info["reply_to_message_id"] = mess["reply_to_message_id"]
#         txt_rep = replace_stock_with_id(txt.strip())
        chat_info["text"]=txt
        chat_list_2.append(chat_info)
#                 print(cht["id_list"])
        print("looooool")
#     is_included,txt = replace_stock_with_id(txt.strip())
    elif is_stock_included:
        chat_info["text"]=txt
        chat_list_2.append(chat_info)
#     elif if_stock_included(txt):
# #         print("if_included_ch",txt)
#         txt_rep = replace_stock_with_id(txt.strip())
#         chat_info["text"]=txt_rep
#         chat_list_2.append(chat_info)
# print(chat_list_1)


CPU times: user 26.8 s, sys: 1.64 s, total: 28.5 s
Wall time: 28.5 s


In [29]:
# chat_list_2
# chat_list_2[1].keys()
len(chat_list_2)
# len(enc.encode(chat_list_2[0]["text"]))
# chat_list_2[0]

30

In [22]:
#@ BOUNDS : [] 
# chat_list
telegram_chanells=[chat_list_2]
# enc = tiktoken.get_encoding("cl100k_base")
# enc.encode("hello world")
channels_limit_bound=[]
for chanell in telegram_chanells:
    chanell_bound=[]
    token_count=0
    i=0
#     for (i,chat) in enumerate(chanell):
    while i < len(chanell):
        chat=chanell[i]
        chat_txt=chat['text']
        chat_tokens_count = len(enc.encode(chat_txt))
        is_more_limit=False
#         token_count+=enc.encode(chat['txt'])
        print(i)
        if( (token_count + chat_tokens_count > 4096) and (not is_more_limit)):
            print( i, len(chanell)  ,  chat_tokens_count,
                  token_count ,token_count + chat_tokens_count > 4096 , i+1 == len(chanell))

            chanell_bound.append(i-1)
            token_count=0
            is_more_limit=True
        elif i+1 == len(chanell):
            chanell_bound.append(i)
            token_count=0
            i+=1
            is_more_limit=False
#             print(chat_txt)
        else:
            token_count += chat_tokens_count
            i+=1
    channels_limit_bound.append(chanell_bound)
channels_limit_bound

0
1
2
3
4
5
6
6 30 98 4016 True False
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29


[[5, 29]]